In [ ]:
import matplotlib
matplotlib.use("Agg")
import numpy as np
import gc
import tensorflow as tf
import keras
import copy
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_regression  

device = "GPU" if tf.config.list_physical_devices("GPU") else "CPU"
print(f"Using Device = {str(device).title()}")
X_raw, y_raw = make_regression(n_samples=500, n_features=6, noise=15.0, random_state=42)

y_raw = y_raw.reshape(-1, 1)
x_scaler = StandardScaler()
y_scaler = StandardScaler()
x = x_scaler.fit_transform(X_raw)
y = y_scaler.fit_transform(y_raw)

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=23)
num_features = x_train.shape[1]

x_train = tf.convert_to_tensor(x_train, dtype=tf.float32)
y_train = tf.convert_to_tensor(y_train, dtype=tf.float32)
x_test = tf.convert_to_tensor(x_test, dtype=tf.float32)
y_test = tf.convert_to_tensor(y_test, dtype=tf.float32)

print(f"X Train = {x_train.shape}")
print(f"Y Train = {y_train.shape}")
print(f"X Test = {x_test.shape}")
print(f"Y Test = {y_test.shape}")

class Regressor(keras.Model):
    def __init__(self, input_dim=num_features):
        super().__init__()

        self.network = keras.Sequential([
            keras.layers.Input(shape=(input_dim,)),
            keras.layers.Dense(8),
            keras.layers.LeakyReLU(negative_slope=0.01),
            keras.layers.BatchNormalization(),

            keras.layers.Dense(16),
            keras.layers.LeakyReLU(negative_slope=0.01),
            keras.layers.BatchNormalization(),

            keras.layers.Dense(28),
            keras.layers.LeakyReLU(negative_slope=0.01),
            keras.layers.BatchNormalization(),

            keras.layers.Dense(47),
            keras.layers.LeakyReLU(negative_slope=0.01),
            keras.layers.BatchNormalization(),
            keras.layers.Dropout(0.7),

            keras.layers.Dense(52),
            keras.layers.ELU(),
            keras.layers.BatchNormalization(),
            keras.layers.Dropout(0.8),

            keras.layers.Dense(68),
            keras.layers.ELU(),
            keras.layers.BatchNormalization(),
            keras.layers.Dropout(0.3),

            keras.layers.Dense(42),
            keras.layers.LeakyReLU(negative_slope=0.01),
            keras.layers.BatchNormalization(),

            keras.layers.Dense(21),
            keras.layers.LeakyReLU(negative_slope=0.01),
            keras.layers.BatchNormalization(),

            keras.layers.Dense(1)
        ])

    def call(self, x, training=False):
        return self.network(x, training=training)


def run_grid_search(rates_list, epochs_list, seed_val):
    best_loss = float("inf")
    best_state = None
    loss_fn = keras.losses.MeanAbsoluteError()

    print(f"\n== Leaderboard ==")
    print(f"{'Learning Rate':<15} | {'Epochs':<8} | {'Val Loss':<10}")
    print(f"=" * 65)
    losses = []

    for lr in rates_list:
        for epoch in epochs_list:
            tf.keras.utils.set_random_seed(int(seed_val))

            model = Regressor()
            model.compile(
                optimizer=keras.optimizers.Adam(learning_rate=lr),
                loss=loss_fn
            )

            model.fit(
                x_train,
                y_train,
                epochs=epoch,
                batch_size=len(x_train),
                verbose=0
            )

            test_loss = model.evaluate(x_test, y_test, verbose=0)
            losses.append(test_loss)

            print(f"{lr:<15} | {epoch:<8} | {test_loss:<10.4f}")

            if test_loss < best_loss:
                best_loss = test_loss
                best_state = copy.deepcopy(model.get_weights())

            del model
            keras.backend.clear_session()
            gc.collect()

    print(f"\n● Best Loss Achieved = {best_loss:<10.4f}")
    print(f"● Worst Loss = {max(losses):<10.4f}")
    return best_state, best_loss, losses

epochs_list = [50, 100]
rates = [0.01, 0.001]

seed = float(input("-- Enter a Value for a Seed : ").strip())
best_state, best_loss, losses = run_grid_search(rates, epochs_list, seed)

fresh_model = Regressor()
fresh_model.build((None, num_features))
fresh_model.set_weights(best_state)

conversion = {
    "Yes": 1,
    "No": 0
}

session_inputs = []
session_predictions = []

while True:
    raw_name_input = input("-- Enter your Name ( or type 'exit' to stop ): ").strip()

    if raw_name_input.lower() == 'exit' or not raw_name_input:
        print("\nExiting Interactive Loop ...")
        break

    user_name = raw_name_input.title()

    hours_studied = float(input("-- How many Hours did you Study for : ").strip())
    attendance_rate = float(input("-- What is your Attendance Rate : ").strip())
    previous_score = float(input("-- What was your Previous Score : ").strip())
    sleep_hours = float(input("-- How many Hours did you Sleep for : ").strip())
    extracurricular_activities = input("-- Do you do any Extra Curricular Activities (Yes/No) : ").strip().title()
    internet_quality = float(input("-- Enter your Internet Quality : ").strip())

    ex = conversion.get(extracurricular_activities, 0)

    raw_array = np.array([[hours_studied, attendance_rate, previous_score, sleep_hours, ex, internet_quality]])
    scaled_array = x_scaler.transform(raw_array)
    scaled_tensor = tf.convert_to_tensor(scaled_array, dtype=tf.float32)

    preds = fresh_model(scaled_tensor, training=False).numpy()

    untransformed = y_scaler.inverse_transform(preds).item()
    print(f"Predicted Performance Score for you, {user_name} = {untransformed:.2f}\n")

    session_inputs.append(f"{user_name}\n({round(hours_studied)} Hours Studied )")
    session_predictions.append(untransformed)

In [ ]:
import tensorflow as tf
import keras
import numpy as np
import gc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.datasets import make_classification  
import copy
from sklearn.metrics import precision_score, recall_score

device = "GPU" if tf.config.list_physical_devices("GPU") else "CPU"
print(f"Using Device = {str(device).title()}")

X_raw, y_raw = make_classification(
    n_samples=500, 
    n_features=6, 
    n_informative=6, 
    n_redundant=0, 
    n_classes=2, 
    random_state=12
)

encoder = LabelEncoder()
y = encoder.fit_transform(y_raw).reshape(-1, 1)
x_scaler = StandardScaler()
x = x_scaler.fit_transform(X_raw)
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=12)
num_features = x_train.shape[1]

x_train = tf.convert_to_tensor(x_train, dtype=tf.float32)
y_train = tf.convert_to_tensor(y_train, dtype=tf.float32)
x_test = tf.convert_to_tensor(x_test, dtype=tf.float32)
y_test = tf.convert_to_tensor(y_test, dtype=tf.float32)

print(f"X Train = {x_train.shape}")
print(f"Y Train = {y_train.shape}")
print(f"X Test = {x_test.shape}")
print(f"Y Test = {y_test.shape}")

class BinaryClassifier(keras.Model):
    def __init__(self, input_dim=num_features):
        super().__init__()

        self.network = keras.Sequential([
            keras.layers.Input(shape=(input_dim,)),

            keras.layers.Dense(8),
            keras.layers.LeakyReLU(negative_slope=0.01),
            keras.layers.BatchNormalization(),

            keras.layers.Dense(16),
            keras.layers.LeakyReLU(negative_slope=0.01),
            keras.layers.BatchNormalization(),

            keras.layers.Dense(28),
            keras.layers.LeakyReLU(negative_slope=0.01),
            keras.layers.BatchNormalization(),

            keras.layers.Dense(47),
            keras.layers.LeakyReLU(negative_slope=0.01),
            keras.layers.BatchNormalization(),
            keras.layers.Dropout(0.7),

            keras.layers.Dense(52),
            keras.layers.ELU(),
            keras.layers.BatchNormalization(),
            keras.layers.Dropout(0.8),

            keras.layers.Dense(68),
            keras.layers.ELU(),
            keras.layers.BatchNormalization(),
            keras.layers.Dropout(0.3),

            keras.layers.Dense(42),
            keras.layers.LeakyReLU(negative_slope=0.01),
            keras.layers.BatchNormalization(),

            keras.layers.Dense(21),
            keras.layers.LeakyReLU(negative_slope=0.01),
            keras.layers.BatchNormalization(),

            keras.layers.Dense(1)
        ])

    def call(self, x, training=False):
        return self.network(x, training=training)


def run_grid_search_binary(rates_list, epochs_list, seed_val):
    best_loss = float("inf")
    best_state = None
    loss_fn = keras.losses.BinaryCrossentropy(from_logits=True)
    losses = []
    accuracies = []
    best_model_weights = None

    print(f"\n== Search Leaderboard ==")
    print(f"{'Learning Rate':<15} | {'Epochs':<8} | {'Val Loss':<10} | {'Val Acc %':<10}")
    print("=" * 65)

    for lr in rates_list:
        for epoch in epochs_list:
            tf.keras.utils.set_random_seed(int(seed_val))

            model = BinaryClassifier()

            model.compile(
                optimizer=keras.optimizers.Adam(learning_rate=lr),
                loss=loss_fn
            )

            model.fit(
                x_train,
                y_train,
                epochs=epoch,
                batch_size=len(x_train),
                verbose=0
            )

            test_logits = model.predict(x_test, verbose=0)
            test_loss = model.evaluate(x_test, y_test, verbose=0)

            test_probabilities = tf.nn.sigmoid(test_logits).numpy()
            test_predictions = (test_probabilities >= 0.5).astype(int)

            test_acc = np.mean(test_predictions.flatten() == y_test.numpy().flatten())

            losses.append(test_loss)
            accuracies.append(test_acc)

            print(f"{lr:<15} | {epoch:<8} | {test_loss:<10.4f} | {test_acc * 100:.2f} %")

            if test_loss < best_loss:
                best_loss = test_loss
                best_state = copy.deepcopy(model.get_weights())
                best_model_weights = copy.deepcopy(model.get_weights())

            del model
            keras.backend.clear_session()
            gc.collect()

    best_model = BinaryClassifier()
    best_model.build((None, num_features))
    best_model.set_weights(best_model_weights)

    best_logits = best_model.predict(x_test, verbose=0)
    best_probabilities = tf.nn.sigmoid(best_logits).numpy()
    best_predictions = (best_probabilities >= 0.5).astype(int)

    print(f"\n● Best Loss Achieved = {best_loss:<10.4f}")
    print(f"● Worst Loss = {max(losses):<10.4f}")
    print(f"● Mean Accuracy = {sum(accuracies) / len(accuracies):<10.4f}")
    print(f"● Precision Score = {precision_score(y_test.numpy(), best_predictions, zero_division=0):.4f}")
    print(f"● Recall Score = {recall_score(y_test.numpy(), best_predictions, zero_division=0):.4f}")

    return best_state, best_loss, losses


# Using a smaller subset of your array list sizes to complete the test loop execution fast
epochs_list = [50, 80]
rates = [0.01, 0.001]
seed = float(input("-- Enter a Value for a Seed : ").strip())

best_state, best_loss, losses = run_grid_search_binary(rates, epochs_list, seed)

fresh_model = BinaryClassifier()
fresh_model.build((None, num_features))
fresh_model.set_weights(best_state)

session_inputs = []
session_predictions = []

statuses = {
    1: "Default",
    0: "Not Default"
}

while True:
    raw_name_input = input("-- Enter Customer Name ( or type 'exit' to stop ): ").strip()

    if raw_name_input.lower() == 'exit' or not raw_name_input:
        print("\nExiting Interactive Loop ...")
        break

    user_name = raw_name_input.title()
    age_val = float(input("-- Enter Age : ").strip())
    income_val = float(input("-- Enter Annual Income : ").strip())
    credit_val = float(input("-- Enter Credit Score : ").strip())
    dti_val = float(input("-- Enter Debt - to - Income Ratio ( 0.0 to 1.0 ) : ").strip())
    missed_val = float(input("-- Enter Missed Payments Last Year : ").strip())
    util_val = float(input("-- Enter Card Utilization Ratio ( 0.0 to 1.0 ) : ").strip())

    raw_array = np.array([[age_val, income_val, credit_val, dti_val, missed_val, util_val]])
    scaled_tensor = tf.convert_to_tensor(x_scaler.transform(raw_array), dtype=tf.float32)

    logits = fresh_model(scaled_tensor, training=False)
    probability = tf.nn.sigmoid(logits).numpy().item()
    pred_class_idx = 1 if probability >= 0.5 else 0

    string_prediction = encoder.inverse_transform([pred_class_idx])[0]

    print(f"Predicted Financial Risk Status = {statuses.get(pred_class_idx, str(string_prediction))}\n")

    session_inputs.append(f"{user_name} ({credit_val} Credit Score)")
    session_predictions.append(probability * 100)

In [ ]:
import numpy as np
import tensorflow as tf
import keras
import gc
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification  
from sklearn.metrics import precision_score, recall_score

device = "GPU" if tf.config.list_physical_devices("GPU") else "CPU"
print(f"Using Device = {str(device).title()}")

X_raw, y_raw = make_classification(
    n_samples=500, 
    n_features=6, 
    n_informative=5, 
    n_redundant=1, 
    n_classes=4, 
    random_state=12
)
encoder = LabelEncoder()
y = encoder.fit_transform(y_raw)
x_scaler = StandardScaler()
x = x_scaler.fit_transform(X_raw)

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=12
)

num_features = x_train.shape[1]
num_classes = len(encoder.classes_)

x_train = tf.convert_to_tensor(x_train, dtype=tf.float32)
y_train = tf.convert_to_tensor(y_train, dtype=tf.int32)
x_test = tf.convert_to_tensor(x_test, dtype=tf.float32)
y_test = tf.convert_to_tensor(y_test, dtype=tf.int32)

print(f"X Train = {x_train.shape}")
print(f"Y Train = {y_train.shape}")
print(f"X Test = {x_test.shape}")
print(f"Y Test = {y_test.shape}")

class MultiClassClassifier(keras.Model):
    def __init__(self, input_dim=num_features, out_dim=num_classes):
        super().__init__()

        self.network = keras.Sequential([
            keras.layers.Input(shape=(input_dim,)),

            keras.layers.Dense(8),
            keras.layers.LeakyReLU(negative_slope=0.01),
            keras.layers.BatchNormalization(),

            keras.layers.Dense(16),
            keras.layers.LeakyReLU(negative_slope=0.01),
            keras.layers.BatchNormalization(),

            keras.layers.Dense(28),
            keras.layers.LeakyReLU(negative_slope=0.01),
            keras.layers.BatchNormalization(),

            keras.layers.Dense(47),
            keras.layers.LeakyReLU(negative_slope=0.01),
            keras.layers.BatchNormalization(),
            keras.layers.Dropout(0.7),

            keras.layers.Dense(52),
            keras.layers.ELU(),
            keras.layers.BatchNormalization(),
            keras.layers.Dropout(0.8),

            keras.layers.Dense(68),
            keras.layers.ELU(),
            keras.layers.BatchNormalization(),
            keras.layers.Dropout(0.3),

            keras.layers.Dense(42),
            keras.layers.LeakyReLU(negative_slope=0.01),
            keras.layers.BatchNormalization(),

            keras.layers.Dense(21),
            keras.layers.LeakyReLU(negative_slope=0.01),
            keras.layers.BatchNormalization(),

            keras.layers.Dense(out_dim)
        ])

    def call(self, x, training=False):
        return self.network(x, training=training)


def run_grid_search_multiclass(rates_list, epochs_list, seed_val):
    best_loss = float("inf")
    best_state = None
    loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    losses = []
    accuracies = []

    print(f"\n== Search Leaderboard ==")
    print(f"{'Learning Rate':<15} | {'Epochs':<8} | {'Val Loss':<10} | {'Val Acc %':<10}")
    print("=" * 65)

    for lr in rates_list:
        for epoch in epochs_list:
            tf.keras.utils.set_random_seed(int(seed_val))

            model = MultiClassClassifier()

            model.compile(
                optimizer=keras.optimizers.Adam(learning_rate=lr),
                loss=loss_fn,
                metrics=[keras.metrics.SparseCategoricalAccuracy()]
            )

            model.fit(
                x_train,
                y_train,
                epochs=epoch,
                batch_size=len(x_train),
                verbose=0
            )

            test_loss, test_acc = model.evaluate(
                x_test, y_test, verbose=0
            )

            losses.append(test_loss)
            accuracies.append(test_acc)

            print(
                f"{lr:<15} | {epoch:<8} | "
                f"{test_loss:<10.4f} | {test_acc * 100:.2f} %"
            )

            if test_loss < best_loss:
                best_loss = test_loss
                best_state = copy.deepcopy(model.get_weights())

            del model
            keras.backend.clear_session()
            gc.collect()

    best_model = MultiClassClassifier()
    best_model.build((None, num_features))
    best_model.set_weights(best_state)

    test_logits = best_model.predict(x_test, verbose=0)
    test_predictions = np.argmax(test_logits, axis=1)

    print(f"\n● Best Loss Achieved = {best_loss:<10.4f}")
    print(f"● Worst Loss = {max(losses):<10.4f}")
    print(f"● Mean Accuracy = {sum(accuracies) / len(accuracies) * 100:.2f} %")

    print(
        f"● Precision Score = {precision_score(y_test.numpy(), test_predictions, average='macro', zero_division=0):.4f}"
    )
    print(
        f"● Recall Score = {recall_score(y_test.numpy(), test_predictions, average='macro', zero_division=0):.4f}\n"
    )

    return best_state, best_loss, losses


epochs_list = [50, 100]
rates = [0.01, 0.001]
seed = float(input("-- Enter a Value for a Seed : ").strip())

best_state, best_loss, losses = run_grid_search_multiclass(
    rates, epochs_list, seed
)

fresh_model = MultiClassClassifier()
fresh_model.build((None, num_features))
fresh_model.set_weights(best_state)

session_inputs = []
session_predictions = []

seg_statuses = {
    0: "Bronze",
    1: "Silver",
    2: "Gold",
    3: "Platinum"
}

while True:
    raw_name_input = input("-- Enter Customer Name ( or type 'exit' to stop ): ").strip()

    if raw_name_input.lower() == 'exit' or not raw_name_input:
        print("\nExiting Interactive Loop ...")
        break

    user_name = raw_name_input.title()
    years_val = float(input("-- Enter Years as a Customer : ").strip())
    spend_val = float(input("-- Enter Annual Spend ($) : ").strip())
    items_val = float(input("-- Enter Total Items Purchased : ").strip())
    return_val = float(input("-- Enter Return Rate (0.0 to 1.0) : ").strip())
    logins_val = float(input("-- Enter Monthly App Logins : ").strip())
    tickets_val = float(input("-- Enter Support Tickets Raised : ").strip())

    raw_array = np.array([[
        years_val,
        spend_val,
        items_val,
        return_val,
        logins_val,
        tickets_val
    ]])

    scaled_tensor = tf.convert_to_tensor(
        x_scaler.transform(raw_array),
        dtype=tf.float32
    )

    logits = fresh_model(scaled_tensor, training=False)
    probability = tf.nn.softmax(logits, axis=1).numpy()
    pred_class_idx = int(np.argmax(probability, axis=1)[0])

    string_prediction = encoder.inverse_transform([pred_class_idx])[0]

    print(
        f"Predicted Customer Segmentation Status = "
        f"{seg_statuses.get(pred_class_idx, str(string_prediction))}\n"
    )

    session_inputs.append(f"{user_name} ( {spend_val} $ Spent )")
    session_predictions.append((probability[0] * 100).tolist())